In [2]:
import os
import pandas as pd

# 1. Establish data paths
input_path = os.path.join('..', 'Data Collection', 'nba_top_100_master.csv')
if not os.path.exists(input_path):
    input_path = 'nba_top_100_master.csv'  # Local fallback

# Load master dataset
df_master = pd.read_csv(input_path)

# List of all 26 required categories
required_columns = [
    'Player_Name', 'Era_ID', 'Era_Label', 'Archetype_ID', 'Archetype_Label',
    'Seasons_Played', 'Career_Games', 'Playoff_Seasons', 'Playoff_Games',
    'Championships', 'Finals_MVP', 'All_NBA_Teams', 'All_Defensive_Teams',
    'Career_PPG', 'Playoff_PPG', 'Career_PER', 'Peak_5Yr_PER', 'Playoff_PER',
    'Career_BPM', 'Playoff_BPM', 'Career_TS_Pct', 'Career_WS_per_48',
    'Playoff_WS_per_48', 'Athletic_Raw_Rank', 'Yahoo_Raw_Rank', 'BR_Raw_Rank', 'All_NBA_PS'
]

# 2. Strict Filter: Remove any player missing data in ANY of these columns
df_modern = df_master.dropna(subset=required_columns).copy()

# 3. Collapse the ranking gaps sequentially for each media list independently
for rank_col in ['Athletic_Raw_Rank', 'Yahoo_Raw_Rank', 'BR_Raw_Rank']:
    df_modern = df_modern.sort_values(by=rank_col)
    df_modern[f'Modern_{rank_col}'] = range(1, len(df_modern) + 1)

# 4. Calculate the new clean modern consensus rank average
df_modern['Modern_Consensus_Rank'] = df_modern[
    ['Modern_Athletic_Raw_Rank', 'Modern_Yahoo_Raw_Rank', 'Modern_BR_Raw_Rank']
].mean(axis=1)

# Sort by final consensus positioning
df_modern = df_modern.sort_values(by='Modern_Consensus_Rank')

# 5. Export to the separate modern database file
output_filename = 'nba_modern_filtered_master.csv'
df_modern.to_csv(output_filename, index=False)
